# Package 설치

In [ ]:
# !uv pip install ipykernel ipywidgets
# !uv pip install transformers tokenizers datasets accelerate sentencepiece pillow  timm -qU

# HuggingFace transformers의 Pipeline을 이용한 모델 활용

- Pipeline은 Transformers 라이브러리의 가장 기본적인 객체로, **전처리 - 추론 -> 후처리** 로 이어지는 일련의 과정을 자동화하여 손쉽게 모델을 사용할 수 있게 해준다.
- Task에 따라 다양한 Pipeline 클래스를 제공하며 `pipeline` 함수를 이용해 쉽게 생성할 수 있다.
- **task만 지정**해서 그 task에 대해 기본적으로 제공 모델과 토크나이저를 사용하거나 또는 **직접 사용할 모델과 토크나이저를 지정**해 생성할 수 있다.
  - **토크나이저의 경우 같은 사용할 모델의 ID로 Load하여 그 모델이 학습할 때 사용한 것을 로드한다.**
- https://huggingface.co/docs/transformers/pipeline_tutorial

![huggingface_pipeline.png](figures/huggingface_pipeline.png)

## 지원하는 주요 태스크
- https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.pipeline.task
### 자연어 처리 태스크
- **text-classification**: 텍스트 분류
- **text-generation**: 텍스트 생성
- **translation**: 번역
- **summarization**: 요약
- **question-answering**: 질의응답
- **fill-mask**: 마스크 토큰 채우기
- **token-classification**: 개체명 인식, Pos tagging 같이 개별 토큰에 대한 분류
- **feature-extraction**: 특징 추출(context vector)

### 영상 처리 태스크
- **image-classification**: 이미지 분류
- **object-detection**
  -  객체 검출 (Object Detection)
  -  이미지 안에서 객체들의 위치와 class를 찾아내는 작업
- **image-segmentation**
  -  이미지 세분화 (Image Segmentation)
  -  이미지를 픽셀 단위로 분할하여 각 픽셀이 어떤 객체에 속하는지 분류하는 작업

## 모델 검색
![huggingface_model_search.png](figures/huggingface_model_search.png)



## pipeline 함수
- 주요파라미터
  - **task:** 수행하려는 작업의 유형을 문자열로 지정한다.
  - **model:**
    - 사용할 사전 학습된 모델의 이름 또는 경로를 지정한다. 
    - 모델이름(ID)은 `[모델소유자이름]/[모델이름]` 형식이다. Hugging Face에서 제공하는 모델의 경우는 `모델소유자이름`이 생략되어 있다. (ex: "google/gemma-2-2b", "gpt2")
    - 모델을 명시적으로 지정하지 않으면, **task에 맞는 기본 모델이 로드**된다.
  - **tokenizer:** 자연어 task에서 사용할 토크나이저를 지정한다. 생략하면 모델과 같이 제공되는(model과 이름이 같은 토크나이저) 토크나이저를 사용한다.
  - **framework:** 사용할 딥러닝 프레임워크를 지정한다. 'pt'는 PyTorch(Default), 'tf'는 TensorFlow를 지정한다.
  - **device:** Pipeline 모델을 실행할 디바이스를 지정한다. 문자열로 `"cpu", "cuda:1", "mps"`, 또는 GPU 번호를 정수로 지정한다. 
  - **revision:** 모델의 특정 버전을 지정할 때 사용한다.
  - **trust_remote_code:** hub 모델을 직접 다운 받는 것이 아니라 모델을 다운 받는 **코드**를 다운 받아 local에서 실행하는 경우 코드를 실행할 수있게 할 지 여부. (bool)
  - **use_fast:** 
    - 빠른 토크나이저를 사용할지 여부를 지정합니다. 기본값은 True입니다.
    - 빠른 토크나이저는 `Rust` 언어로 구현되어 속도가 빠르다. 단 모든 모델에 대해 지원하지 않는다. 지원하지 않을 경우 `use_fast=True`로 설정해도 일반 토크나이저가 사용된다.

In [1]:
import transformers

transformers.__version__

'4.57.3'

## Task 별 pipeline 실습

### 텍스트 분류

In [2]:
from transformers import pipeline

# task만 지정: 그 task를 실행할 수 있는 기본 모델과 토크나이저(전처러기)를 이용해서 pipeline 생성
pipe = pipeline(task="text-classification")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


In [3]:
result = pipe("I am very happy")

In [4]:
result

[{'label': 'POSITIVE', 'score': 0.9998795986175537}]

In [5]:
data = [ 
    "The project was completed successfully.", 
    "She always brings positive energy to the team.", 
    "I am confident that we will achieve our goals.",
    "The results were not as expected.", 
    "He struggled to meet the deadline.", 
    "The client was dissatisfied with the final product." 
]

In [6]:
result = pipe(data)

In [7]:
result

[{'label': 'POSITIVE', 'score': 0.9998227953910828},
 {'label': 'POSITIVE', 'score': 0.9998812675476074},
 {'label': 'POSITIVE', 'score': 0.9998470544815063},
 {'label': 'NEGATIVE', 'score': 0.9978100657463074},
 {'label': 'NEGATIVE', 'score': 0.99960857629776},
 {'label': 'NEGATIVE', 'score': 0.9996129870414734}]

In [8]:
# pipeline 생성 시 모델을 지정
# 1. model_id를 지정 - 지정한 ID의 모델과 토크나이저로 pipeline을 구성
# 2. Model과 Tokenizer(전처리)를 직접 생성해서 pipeline에 넣어서 생성.

model = 'distilbert/distilbert-base-uncased-finetuned-sst-2-english'
pipe = pipeline(
    task='text-classification',
    model=model,
    # tokenizer=model # tokenizer와 model의 id가 같은 경우 생략
)

Device set to use cpu


In [9]:
kor_texts = [
    "이 영화 정말 재미있어요!",
    "서비스가 별로였어요.",
    "제품 품질이 우수합니다.",
    "따듯하고 부드럽고 제품은 너무 좋습니다. 그런데 배송이 너무 늦네요."
]

In [10]:
pipe(kor_texts)

[{'label': 'POSITIVE', 'score': 0.9855567812919617},
 {'label': 'POSITIVE', 'score': 0.7425776124000549},
 {'label': 'POSITIVE', 'score': 0.6555716395378113},
 {'label': 'NEGATIVE', 'score': 0.5247918367385864}]

In [11]:
model = 'Copycats/koelectra-base-v3-generalized-sentiment-analysis'
pipe = pipeline(
    task='text-classification',
    model=model,
)

Device set to use cpu


In [12]:
pipe(kor_texts)

[{'label': '1', 'score': 0.9897311329841614},
 {'label': '0', 'score': 0.9969298243522644},
 {'label': '1', 'score': 0.9640172123908997},
 {'label': '0', 'score': 0.5669127702713013}]

### 제로샷 분류
- 제로샷(Zero-shot)은 각 개별 작업에 대한 특정 교육 없이 작업을 수행할 수 있는 task다.
- 입력 텍스트와 함께 클래스 레이블을 제공하면 분류 작업을 한다.
- 모델은  `task`에서 `Zero-Shot` 으로 시작하는 task를 선택하여 검색한다.

In [13]:
model = 'facebook/bart-large-mnli'
pipe = pipeline(
    task='zero-shot-classification',
    model=model,
)

Device set to use cpu


In [14]:
# 분류 대상
text = ["Python is a programming language.", 
        "I love soccer", 
        "The stock price rose slightly today."]

# 분류 클래스
labels1 = ["IT", "Sports"]
labels2 = ["business", "programming", "sports", "movie", "education"]

In [15]:
result = pipe(text, candidate_labels=labels1)
result

[{'sequence': 'Python is a programming language.',
  'labels': ['IT', 'Sports'],
  'scores': [0.5758535265922546, 0.4241464138031006]},
 {'sequence': 'I love soccer',
  'labels': ['Sports', 'IT'],
  'scores': [0.9935312867164612, 0.006468690931797028]},
 {'sequence': 'The stock price rose slightly today.',
  'labels': ['IT', 'Sports'],
  'scores': [0.6849520802497864, 0.3150479197502136]}]

In [16]:
result = pipe(text, candidate_labels=labels2)
result

[{'sequence': 'Python is a programming language.',
  'labels': ['programming', 'business', 'movie', 'sports', 'education'],
  'scores': [0.9856367111206055,
   0.005072721280157566,
   0.0034023483749479055,
   0.002961924998089671,
   0.0029262355528771877]},
 {'sequence': 'I love soccer',
  'labels': ['sports', 'programming', 'business', 'movie', 'education'],
  'scores': [0.9952405691146851,
   0.0012840895215049386,
   0.0012676474871113896,
   0.0012649551499634981,
   0.0009427034528926015]},
 {'sequence': 'The stock price rose slightly today.',
  'labels': ['business', 'movie', 'programming', 'sports', 'education'],
  'scores': [0.7462778091430664,
   0.06974831968545914,
   0.06889291107654572,
   0.0645080953836441,
   0.05057287961244583]}]

### 텍스트 생성

In [17]:
pipe = pipeline(task='text-generation')

No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d (https://huggingface.co/openai-community/gpt2).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [18]:
result = pipe('Python is a')

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [19]:
result

[{'generated_text': 'Python is a cross platform language that is designed to be flexible and easy-to-use.\n\nThe following is a list of some of the most popular languages that are available in the GNU/Linux kernel.\n\n1. UNIX\n\nThe GNU/Linux kernel includes two distinct versions of UNIX. The GNU/Linux kernel is distributed along with the GNU/Linux operating system.\n\n2. OS X\n\nThe OS X operating system includes three versions of UNIX: the X Window System (X.Y.W.W.), the X Window System (X.Y.W.W.), and the X Window System (X.Y.W.W.).\n\n3. Linux\n\nThe Linux operating system includes three versions of UNIX:\n\nUbuntu\n\nThe Unix-like Linux operating system has an extensive set of features to support the operating system, including support for the following:\n\nThe ability to run as a web browser\n\nThe ability to open files and directories, as well as the ability to read and write files\n\nThe ability to read and write files and directories, as well as the ability to read and write f

In [20]:
print(result[0]['generated_text'])

Python is a cross platform language that is designed to be flexible and easy-to-use.

The following is a list of some of the most popular languages that are available in the GNU/Linux kernel.

1. UNIX

The GNU/Linux kernel includes two distinct versions of UNIX. The GNU/Linux kernel is distributed along with the GNU/Linux operating system.

2. OS X

The OS X operating system includes three versions of UNIX: the X Window System (X.Y.W.W.), the X Window System (X.Y.W.W.), and the X Window System (X.Y.W.W.).

3. Linux

The Linux operating system includes three versions of UNIX:

Ubuntu

The Unix-like Linux operating system has an extensive set of features to support the operating system, including support for the following:

The ability to run as a web browser

The ability to open files and directories, as well as the ability to read and write files

The ability to read and write files and directories, as well as the ability to read and write files

The ability to write files and director

In [21]:
pipe('나는 어제')

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': '나는 어제는 들아어오는 들아어오는 게아도 지바길 만선은 아는 차리는 게아도 첩아도 첩아도 게아도 에아도 더게도 조나는 게아도 더게도 조나는 게아도 조나는 게아도 조나는 게아도 조나는 게아도 게아도 조나는 �'}]

In [23]:
model = 'Qwen/Qwen3-0.6B'

pipe=pipeline('text-generation', model=model)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Device set to use cpu


In [24]:
result = pipe("나는 어제")

In [25]:
print(result[0]["generated_text"])

나는 어제 일어나 일을 시작했고, 그 이후에 몇 주 동안 루트의 일과 관련된 일은 없었고, 그리고 오늘은 공식적으로 발표를 했고, 그 발표는 학교의 교육 목적에 대한 설명을 했고, 그리고 그 발표는 학교의 학교의 교육 목적에 대한 설명을 했고, 그리고 그 발표는 학교의 교육 목적에 대한 설명을 했고, 그리고 그 발표는 학교의 교육 목적에 대한 설명을 했고, 그리고 그 발표는 학교의 교육 목적에 대한 설명을 했고, 그리고 그 발표는 학교의 교육 목적에 대한 설명을 했고, 그리고 그 발표는 학교의 교육 목적에 대한 설명을 했고, 그리고 그 발표는 학교의 교육 목적에 대한 설명을 했고, 그리고 그 발표는 학교의 교육 목적에 대한 설명을 했고, 그리고 그 발표는 학교의 교육 목적에 대한 설명을 했고, 그리고 그 발표는 학교의


### 마스크 채우기

In [26]:
# 모델들마다 mask 모양이 다르다. 모델에 맞게 채워줘야한다.
text = "I'm going to <mask> because <mask> am hurt."

In [27]:
model = 'FacebookAI/xlm-roberta-large'
pipe = pipeline(task='fill-mask',model=model)

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of the model checkpoint at FacebookAI/xlm-roberta-large were not used when initializing XLMRobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [28]:
result = pipe(text)
result

[[{'score': 0.3051992654800415,
   'token': 91190,
   'token_str': 'cry',
   'sequence': "<s> I'm going to cry because<mask> am hurt.</s>"},
  {'score': 0.07201600074768066,
   'token': 68,
   'token_str': 'die',
   'sequence': "<s> I'm going to die because<mask> am hurt.</s>"},
  {'score': 0.041744109243154526,
   'token': 31358,
   'token_str': 'leave',
   'sequence': "<s> I'm going to leave because<mask> am hurt.</s>"},
  {'score': 0.03971703723073006,
   'token': 33022,
   'token_str': 'write',
   'sequence': "<s> I'm going to write because<mask> am hurt.</s>"},
  {'score': 0.028799382969737053,
   'token': 60268,
   'token_str': 'sleep',
   'sequence': "<s> I'm going to sleep because<mask> am hurt.</s>"}],
 [{'score': 0.9688465595245361,
   'token': 87,
   'token_str': 'I',
   'sequence': "<s> I'm going to<mask> because I am hurt.</s>"},
  {'score': 0.02960127405822277,
   'token': 17,
   'token_str': 'i',
   'sequence': "<s> I'm going to<mask> because i am hurt.</s>"},
  {'score'

In [29]:
len(result)

2

In [31]:
len(result[0])  # 첫번쨰 <mask> 에 대한 추론결과. 5: top-5
for r in result[0]:
    print(r)

{'score': 0.3051992654800415, 'token': 91190, 'token_str': 'cry', 'sequence': "<s> I'm going to cry because<mask> am hurt.</s>"}
{'score': 0.07201600074768066, 'token': 68, 'token_str': 'die', 'sequence': "<s> I'm going to die because<mask> am hurt.</s>"}
{'score': 0.041744109243154526, 'token': 31358, 'token_str': 'leave', 'sequence': "<s> I'm going to leave because<mask> am hurt.</s>"}
{'score': 0.03971703723073006, 'token': 33022, 'token_str': 'write', 'sequence': "<s> I'm going to write because<mask> am hurt.</s>"}
{'score': 0.028799382969737053, 'token': 60268, 'token_str': 'sleep', 'sequence': "<s> I'm going to sleep because<mask> am hurt.</s>"}


In [32]:
kor_text = "오늘 밤은 전국이 흐린 가운데 대부분 지역에 <mask>가 내리겠고, 기온이 내려가면서 점차 <mask>이 오는 곳이 많겠습니다"

In [34]:
result = pipe(kor_text)
result

[[{'score': 0.9305621981620789,
   'token': 7091,
   'token_str': '비',
   'sequence': '<s> 오늘 밤은 전국이 흐린 가운데 대부분 지역에 비 가 내리겠고, 기온이 내려가면서 점차<mask> 이 오는 곳이 많겠습니다</s>'},
  {'score': 0.04358893632888794,
   'token': 34565,
   'token_str': '눈',
   'sequence': '<s> 오늘 밤은 전국이 흐린 가운데 대부분 지역에 눈 가 내리겠고, 기온이 내려가면서 점차<mask> 이 오는 곳이 많겠습니다</s>'},
  {'score': 0.0054661487229168415,
   'token': 208400,
   'token_str': '구름',
   'sequence': '<s> 오늘 밤은 전국이 흐린 가운데 대부분 지역에 구름 가 내리겠고, 기온이 내려가면서 점차<mask> 이 오는 곳이 많겠습니다</s>'},
  {'score': 0.0024038865230977535,
   'token': 6452,
   'token_str': '시',
   'sequence': '<s> 오늘 밤은 전국이 흐린 가운데 대부분 지역에 시 가 내리겠고, 기온이 내려가면서 점차<mask> 이 오는 곳이 많겠습니다</s>'},
  {'score': 0.001108957570977509,
   'token': 1504,
   'token_str': '이',
   'sequence': '<s> 오늘 밤은 전국이 흐린 가운데 대부분 지역에 이 가 내리겠고, 기온이 내려가면서 점차<mask> 이 오는 곳이 많겠습니다</s>'}],
 [{'score': 0.7088543772697449,
   'token': 34565,
   'token_str': '눈',
   'sequence': '<s> 오늘 밤은 전국이 흐린 가운데 대부분 지역에<mask> 가 내리겠고, 기온이 내려가면서 점차 눈 이 오는 곳이 많

In [36]:
result = pipe(kor_text, top_k=2) # top_k 기본값 :5
result

[[{'score': 0.9305621981620789,
   'token': 7091,
   'token_str': '비',
   'sequence': '<s> 오늘 밤은 전국이 흐린 가운데 대부분 지역에 비 가 내리겠고, 기온이 내려가면서 점차<mask> 이 오는 곳이 많겠습니다</s>'},
  {'score': 0.04358893632888794,
   'token': 34565,
   'token_str': '눈',
   'sequence': '<s> 오늘 밤은 전국이 흐린 가운데 대부분 지역에 눈 가 내리겠고, 기온이 내려가면서 점차<mask> 이 오는 곳이 많겠습니다</s>'}],
 [{'score': 0.7088543772697449,
   'token': 34565,
   'token_str': '눈',
   'sequence': '<s> 오늘 밤은 전국이 흐린 가운데 대부분 지역에<mask> 가 내리겠고, 기온이 내려가면서 점차 눈 이 오는 곳이 많겠습니다</s>'},
  {'score': 0.23191769421100616,
   'token': 7091,
   'token_str': '비',
   'sequence': '<s> 오늘 밤은 전국이 흐린 가운데 대부분 지역에<mask> 가 내리겠고, 기온이 내려가면서 점차 비 이 오는 곳이 많겠습니다</s>'}]]

### Token별 분류
- task: token-classification 
  - 개체명인식(ner), 품사부착(pos tagging)을 수행하는 task 
  - 개체명 인식은 문장에서 특정한 개체명(예: 사람 이름, 지명, 조직명 등)을 식별하는 task이다. 

In [37]:
text = "My name is Sylvain and I work at Hugging Face in Brooklyn."

In [38]:
pipe = pipeline(task='token-classification', model = 'dbmdz/bert-large-cased-finetuned-conll03-english')
result = pipe(text)

config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


In [39]:
result

[{'entity': 'I-PER',
  'score': np.float32(0.99938285),
  'index': 4,
  'word': 'S',
  'start': 11,
  'end': 12},
 {'entity': 'I-PER',
  'score': np.float32(0.99815494),
  'index': 5,
  'word': '##yl',
  'start': 12,
  'end': 14},
 {'entity': 'I-PER',
  'score': np.float32(0.99590707),
  'index': 6,
  'word': '##va',
  'start': 14,
  'end': 16},
 {'entity': 'I-PER',
  'score': np.float32(0.99923277),
  'index': 7,
  'word': '##in',
  'start': 16,
  'end': 18},
 {'entity': 'I-ORG',
  'score': np.float32(0.9738931),
  'index': 12,
  'word': 'Hu',
  'start': 33,
  'end': 35},
 {'entity': 'I-ORG',
  'score': np.float32(0.976115),
  'index': 13,
  'word': '##gging',
  'start': 35,
  'end': 40},
 {'entity': 'I-ORG',
  'score': np.float32(0.9887976),
  'index': 14,
  'word': 'Face',
  'start': 41,
  'end': 45},
 {'entity': 'I-LOC',
  'score': np.float32(0.9932106),
  'index': 16,
  'word': 'Brooklyn',
  'start': 49,
  'end': 57}]

### 질의 응답
- 문서와 질문을 주면 문서에서 답을 찾아 응답한다.

In [40]:
pipe=pipeline(task='question-answering', model='timpal0l/mdeberta-v3-base-squad2')

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/453 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Device set to use cpu


In [44]:
# question="Where do I work?"
question="Where is Hugging Face?"
context="My name is Sylvain and I work at Hugging Face in Brooklyn"

In [45]:
result = pipe(question=question, context=context)
result

{'score': 0.9884676337242126, 'start': 48, 'end': 57, 'answer': ' Brooklyn'}

In [46]:
context = """우리나라 2대 수출 품목인 자동차가 도널드 트럼프 미국 행정부의 관세 여파로 지난달 큰 폭의 수출 감소율을 보이면서 우려가 커지고 있다. 현대차, 기아의 미국 수출 비중이 최대 85%에 이르는 상황에서 자동차 관세 장기화 시 피해는 걷잡을 수 없이 불어날 것이라는 암울한 전망이 나온다.
1일 산업통상자원부가 발표한 5월 수출입 동향에 따르면 지난달 자동차 수출은 작년 동기 대비 4.4% 감소한 62억달러로 집계됐다. 최대 자동차 시장인 미국으로의 수출은 18억4000만달러로 무려 32.0% 급감했다.
4월 미국의 수입산 자동차 25% 관세 부과에 이어 5월부터 일부 자동차 부품에도 25%의 관세가 적용된 결과다. 관세 장기화 시 피해는 더 커질 것이라는 우려가 현실화한 셈이다.
국내 완성차 1·2위 업체인 현대차·기아는 현지 생산 비중을 확대하는 동시에 가격 인상을 검토하고 있다. 관세 여파를 흡수하기 위해서다. 가격 인상이 현실화할 경우 미국 현지 판매는 줄어들 수밖에 없어 수출에는 더 악영향을 미칠 것으로 보인다.
"""

q1 = "현대차 기아의 미국 수출비중은?"
q2 = "자동차 수출이 얼마나 급감했나?"
q3 = "대미 수출 감소에 국내 자동차 업체들의 대응방법은?"

In [48]:
result = pipe(question=[q1, q2, q3], context = context)
result

[{'score': 0.4708716640016064, 'start': 95, 'end': 103, 'answer': ' 최대 85%에'},
 {'score': 0.9133416299591772, 'start': 270, 'end': 276, 'answer': ' 32.0%'},
 {'score': 0.27349893003702164,
  'start': 426,
  'end': 442,
  'answer': ' 가격 인상을 검토하고 있다.'}]

In [49]:
len(result)

3

In [50]:
for r in result:
    print(r)

{'score': 0.4708716640016064, 'start': 95, 'end': 103, 'answer': ' 최대 85%에'}
{'score': 0.9133416299591772, 'start': 270, 'end': 276, 'answer': ' 32.0%'}
{'score': 0.27349893003702164, 'start': 426, 'end': 442, 'answer': ' 가격 인상을 검토하고 있다.'}


### 문서 요약

In [51]:
model = 'eenzeenee/t5-base-korean-summarization'
pipe=pipeline(task='summarization', model=model)

config.json:   0%|          | 0.00/782 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [52]:
result = pipe(context)

Token indices sequence length is longer than the specified maximum sequence length for this model (368 > 128). Running this sequence through the model will result in indexing errors


In [53]:
result[0]['summary_text']

'자동차가 트럼프 미국 행정부의 관세 여파로 큰 폭의 수출 감소율을 보이면서 자동차 관세 장기화 시 피해는 걷잡을 수 없이 불어날 것이라는 암울한 전망이 나온다.'

### 번역

In [ ]:
text = "Ce cours est produit par Hugging Face."

In [55]:
model='Helsinki-NLP/opus-mt-fr-en'
pipe=pipeline(task='translation', model=model)

Device set to use cpu


In [56]:
result = pipe(text)
result

[{'translation_text': 'My name is Sylvain and I work at Hugging Face in Brooklyn.'}]

In [57]:
text_list = ["이 문장을 영어로 번역합니다.", "날씨가 점점 더워집니다.", "오늘 비가 올 것 같습니다."]

In [60]:
model='Helsinki-NLP/opus-mt-ko-en'
pipe=pipeline(task='translation', model=model)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/842k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [61]:
result = pipe(text_list)
result

[{'translation_text': 'I translate this sentence into English.'},
 {'translation_text': 'The weather gets warmer and warmer.'},
 {'translation_text': "It's going to rain today."}]

### 이미지를 설명하는 텍스트 생성

In [62]:
url1 = "https://huggingface.co/datasets/Narsil/image_dummy/resolve/main/parrots.png"
url2 = "https://th.bing.com/th?id=ORMS.c526884bbea37c0bb9501f4f83b601e4&pid=Wdp&w=268&h=140&qlt=90&c=1&rs=1&dpr=1&p=0"
url3 = "http://images.cocodataset.org/val2017/000000039769.jpg"

In [65]:
model='Salesforce/blip-image-captioning-base'
pipe=pipeline(task='image-to-text', model=model)

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cpu


In [67]:
result = pipe(url1)
result

[{'generated_text': 'two birds are standing next to each other birds'}]

In [68]:
result = pipe([url1, url2, url3])
result

[[{'generated_text': 'two birds are standing next to each other birds'}],
 [{'generated_text': 'a baseball player is throwing a pitch'}],
 [{'generated_text': 'two cats sleeping on a couch'}]]

In [74]:
# 로컬 컴퓨터의 이미지: 파일경로
pipe(['data/whale.jpg'])

[[{'generated_text': 'a humpback breaching in the ocean'}]]

### 이미지 분류

In [75]:
url = "https://pds.joongang.co.kr/news/component/htmlphoto_mmdata/202306/25/488f9638-800c-4bac-ad65-82877fbff79b.jpg"

In [76]:
model='google/vit-base-patch16-224'
pipe=pipeline(task='image-classification', model=model)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Device set to use cpu


In [77]:
result = pipe(url)
result

[{'label': 'Egyptian cat', 'score': 0.8531318306922913},
 {'label': 'tabby, tabby cat', 'score': 0.047503840178251266},
 {'label': 'tiger cat', 'score': 0.03486618027091026},
 {'label': 'Persian cat', 'score': 0.007555845659226179},
 {'label': 'Siamese cat, Siamese', 'score': 0.0037885899655520916}]

In [95]:
result = pipe([url, "data/whale.jpg"], top_k=2)
for r in result:
    print(r)
    print('====================')

[{'score': 0.9452526569366455, 'label': 'cat', 'box': {'xmin': 0, 'ymin': 15, 'xmax': 483, 'ymax': 511}}, {'score': 0.8915566802024841, 'label': 'cat', 'box': {'xmin': 109, 'ymin': 15, 'xmax': 485, 'ymax': 511}}]
[{'score': 0.5347470045089722, 'label': 'boat', 'box': {'xmin': 93, 'ymin': 47, 'xmax': 581, 'ymax': 430}}]


### Object Detection

In [88]:
image_path1 = r"data/image1.jpg"
image_path2 = r"data/image2.jpg"
image_path3 = r"data/image3.jpg"

In [89]:
pipe = pipeline(task='object-detection')

No model was supplied, defaulted to facebook/detr-resnet-50 and revision 1d5f47b (https://huggingface.co/facebook/detr-resnet-50).
Using a pipeline without specifying a model name and revision in production is not recommended.
c:\Users\82107\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\module.py:2446: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
c:\Users\82107\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\module.py:2446: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in th

In [90]:
result  = pipe(image_path1)
result

[{'score': 0.9988904595375061,
  'label': 'dog',
  'box': {'xmin': 430, 'ymin': 423, 'xmax': 533, 'ymax': 597}},
 {'score': 0.9998466968536377,
  'label': 'person',
  'box': {'xmin': 531, 'ymin': 158, 'xmax': 673, 'ymax': 581}}]

In [93]:
result = pipe([image_path1, image_path2, image_path3])

for r in result:
    print(r)
    print('-----------')

[{'score': 0.9988904595375061, 'label': 'dog', 'box': {'xmin': 430, 'ymin': 423, 'xmax': 533, 'ymax': 597}}, {'score': 0.9998466968536377, 'label': 'person', 'box': {'xmin': 531, 'ymin': 158, 'xmax': 673, 'ymax': 581}}]
-----------
[{'score': 0.9981694221496582, 'label': 'cat', 'box': {'xmin': 541, 'ymin': 122, 'xmax': 719, 'ymax': 535}}, {'score': 0.9980827569961548, 'label': 'cat', 'box': {'xmin': 198, 'ymin': 48, 'xmax': 373, 'ymax': 459}}, {'score': 0.9971736669540405, 'label': 'cat', 'box': {'xmin': 0, 'ymin': 89, 'xmax': 255, 'ymax': 535}}, {'score': 0.8180071115493774, 'label': 'bench', 'box': {'xmin': 218, 'ymin': 355, 'xmax': 718, 'ymax': 535}}, {'score': 0.9972655773162842, 'label': 'cat', 'box': {'xmin': 366, 'ymin': 60, 'xmax': 580, 'ymax': 479}}]
-----------
[{'score': 0.9956638216972351, 'label': 'cell phone', 'box': {'xmin': 96, 'ymin': 165, 'xmax': 136, 'ymax': 236}}, {'score': 0.9919518232345581, 'label': 'tv', 'box': {'xmin': 147, 'ymin': 28, 'xmax': 429, 'ymax': 240}